In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import transformers
import torch

In [ ]:
# Authenticate to Hugging Face Hub
from huggingface_hub import login
login()


In [ ]:
device = "cuda"  # Use CUDA (GPU) for faster computations, if available

# Configuration for loading the Q/A Large Language Model (LLM) with 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Load the model in 4-bit precision to reduce memory usage and increase efficiency
    bnb_4bit_use_double_quant=True,  # Enable double quantization for better accuracy in 4-bit mode
    bnb_4bit_quant_type="nf4",  # Use NormalFloat4 (nf4) quantization for optimized model performance
    bnb_4bit_compute_dtype=torch.bfloat16  # Use bfloat16 precision for computation to balance speed and memory
)


In [ ]:
# Load the Q&A LLM
model_id = "google/gemma-7b"
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"": 0})
tokenizer = AutoTokenizer.from_pretrained(model_id, add_eos_token=True, padding_side='left')

# Load the fine-tuned emotion prediction model from Hugging Face
model_id2 = "oumiii/gemma-7b-Finetune-test_wh"
Emo_model = AutoModelForCausalLM.from_pretrained(model_id2, quantization_config=bnb_config, device_map=device)
Emo_tokenizer = AutoTokenizer.from_pretrained(model_id2, add_eos_token=True, padding_side='left')

In [ ]:
#structure conversation 
# Create an empty list to store conversation dictionaries
conversations = []
LLM_Emotion=[]


# Initialize variables to keep track of the current dialogue ID
current_dialogue_id = None
current_conv = {}
text=""

def create_conversation(text,query,result):
        
        text=text+"\nHuman:"+query+"\nAima:"+result
        return text


In [ ]:
def predict_emotion(text,model,tokenizer):

        prompt_template = """
        <bos>You should give the emotion that supposed to be expressed by the last speaker of the conversation based on the utterences and the previous emotions of the speakers(human and Aima).\nThe emotion should be one of these emotions : ""joy (1)"", ""sadness (2)"", ""fear (3)"", ""disgust (4)"", ""surprise (5)"" or ""neutral (6)"" or ""anger (7)"".\nGive the emotion of the last speaker of this conversation without any explainations:
        {query}
        <eos><eos>
        """
        prompt = prompt_template.format(query=text)

        encodeds = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)

        model_inputs = encodeds.to(device)

        generated_ids = model.generate(**model_inputs, max_new_tokens=10, do_sample=True,temperature=0.1)

        decoded = tokenizer.batch_decode(generated_ids)
        result=decoded[0]

        start_index = result.find("<eos><eos>")
        desired_text = result[start_index:].strip() 
        # print(desired_text)   

        def detect_words(text, word_list):
            text_lower = text.lower()
            # text=text+f"({detected_words[0]})"
            detected_words = [word for word in word_list if word in text_lower]
            return detected_words

        word_list = ['neutral','surprise','fear','sadness','joy','disgust','anger']
        detected_words = detect_words(desired_text, word_list)

        if detected_words:
            text=text+f"({detected_words[0]})"
        else:
            print("No emotion detected")
        return detected_words[0],text


In [ ]:
def get_completion(query: str, model, tokenizer) -> str:
    prompt_template = """<s>[INST]You are an AI assistant Aima, you help me whenever I need help, you should speak a human not like an AI!
  {query}
  [/INST]
  </s>
  """
    prompt = prompt_template.format(query=query)

    encodeds = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)

    model_inputs = encodeds.to(device)

    generated_outputs = model.generate(**model_inputs, max_new_tokens=100, do_sample=True, return_dict_in_generate=True, output_scores=False)
    generated_ids = generated_outputs['sequences']

    # Decode the generated tokens excluding the prompt part
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # Remove the prompt from the generated text
    prompt_end = prompt.rfind("[/INST]\n  </s>")
    if prompt_end != -1:
        generated_text = generated_text[prompt_end + len("[/INST]\n  </s>"):].strip()

    return generated_text


In [ ]:
while True:
    query=input("let's start a conversation:")
    result = get_completion(query, model=model, tokenizer=tokenizer)
    print(result)
    conversation=create_conversation(text,query,result)
    emotion,conversation=predict_emotion(conversation,model=Emo_model,tokenizer=Emo_tokenizer)
    print(result)
    print(emotion)